# 04 — Trace Analysis

Analyze execution traces: synthetic vs real, path coverage,
intent diversity, and learning progression over time.

**Requires:** `vault-exec init` + some `vault-exec run` executions

In [1]:
import { openVaultStore } from "../src/db/index.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const db = await openVaultStore(DB_PATH);
const traces = await db.getAllTraces();
const allNotes = await db.getAllNotes();
const noteNames = new Set(allNotes.map(n => n.name));

console.log(`Total traces: ${traces.length}`);
console.log(`Notes in vault: ${noteNames.size}`);


Total traces: 28


Notes in vault: 16


In [2]:
// Trace overview
const synthetic = traces.filter(t => t.synthetic);
const real = traces.filter(t => !t.synthetic);
const withIntent = traces.filter(t => t.intent);
const withIntentEmb = traces.filter(t => t.intentEmbedding && t.intentEmbedding.length > 0);

console.log('\n=== Trace Overview ===');
console.log(`Synthetic:          ${synthetic.length}`);
console.log(`Real:               ${real.length}`);
console.log(`With intent text:   ${withIntent.length}`);
console.log(`With intent embed:  ${withIntentEmb.length}`);

// Unique targets covered
const synTargets = new Set(synthetic.map(t => t.targetNote));
const realTargets = new Set(real.map(t => t.targetNote));
const allTargets = new Set(traces.map(t => t.targetNote));

console.log(`\nTarget coverage:`);
console.log(`  Synthetic targets: ${synTargets.size}/${noteNames.size}`);
console.log(`  Real targets:      ${realTargets.size}/${noteNames.size}`);
console.log(`  Total coverage:    ${allTargets.size}/${noteNames.size}`);

// Uncovered notes
const uncovered = [...noteNames].filter(n => !allTargets.has(n));
if (uncovered.length > 0) {
  console.log(`\n⚠ Uncovered notes (no trace targets them):`);
  uncovered.forEach(n => console.log(`  - ${n}`));
}


=== Trace Overview ===


Synthetic:          8


Real:               20


With intent text:   27


With intent embed:  19


...

  - NPS Snapshot


  - Projects


  - Renewal Watchlist


  - Support Tickets


  - Upsell Candidates


In [3]:
// Trace type breakdown
const traceTypeData = [
  { type: 'Synthetic', count: synthetic.length },
  { type: 'Real (with intent)', count: real.filter(t => t.intent).length },
  { type: 'Real (no intent)', count: real.filter(t => !t.intent).length },
];

const typeSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Trace Types",
  "width": 300,
  "height": 200,
  "data": { "values": traceTypeData },
  "mark": { "type": "arc", "innerRadius": 50 },
  "encoding": {
    "theta": { "field": "count", "type": "quantitative" },
    "color": { "field": "type", "type": "nominal", "scale": { "range": ["#90CAF9", "#4CAF50", "#FF9800"] } },
    "tooltip": [{ "field": "type" }, { "field": "count" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": typeSpec }, { raw: true });

In [4]:
// Path analysis: execution paths and their frequency
const pathCounts = new Map<string, number>();
for (const t of traces) {
  const pathStr = t.path.join(' → ');
  pathCounts.set(pathStr, (pathCounts.get(pathStr) ?? 0) + 1);
}

console.log('\n=== Execution Paths ===');
[...pathCounts.entries()]
  .sort((a, b) => b[1] - a[1])
  .forEach(([path, count]) => {
    console.log(`  ${count}x  ${path}`);
  });

// Path length distribution
const pathLengths = traces.map(t => ({ length: t.path.length, synthetic: t.synthetic }));
const avgLen = pathLengths.reduce((s, p) => s + p.length, 0) / pathLengths.length;
console.log(`\nAvg path length: ${avgLen.toFixed(1)}`);
console.log(`Max path length: ${Math.max(...pathLengths.map(p => p.length))}`);


=== Execution Paths ===


  5x  Employees → Department Budget


  4x  Employees → Projects → Active Projects → Project Costs → Over Budget


  4x  Employees


  3x  Projects → Active Projects → Biggest Team


...

  1x  Hiring Need → Project Costs → Department Budget → Over Budget


  1x  Hiring Need → Department Budget → Over Budget → Project Costs


  1x  NPS Snapshot → Renewal Watchlist → Revenue Risk Score → Support Tickets → Action Queue



Avg path length: 3.1


Max path length: 5


In [5]:
// Path length distribution chart
const lenCounts = new Map<number, {synthetic: number, real: number}>();
for (const t of traces) {
  const len = t.path.length;
  if (!lenCounts.has(len)) lenCounts.set(len, { synthetic: 0, real: 0 });
  if (t.synthetic) lenCounts.get(len)!.synthetic++;
  else lenCounts.get(len)!.real++;
}

const lenData: Array<{length: number, count: number, type: string}> = [];
for (const [len, counts] of lenCounts) {
  lenData.push({ length: len, count: counts.synthetic, type: 'synthetic' });
  lenData.push({ length: len, count: counts.real, type: 'real' });
}

const lenSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Path Length Distribution",
  "width": 400,
  "height": 250,
  "data": { "values": lenData },
  "mark": "bar",
  "encoding": {
    "x": { "field": "length", "type": "ordinal", "title": "Path Length" },
    "y": { "field": "count", "type": "quantitative", "title": "Traces", "stack": true },
    "color": { "field": "type", "type": "nominal" },
    "tooltip": [{ "field": "length" }, { "field": "type" }, { "field": "count" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": lenSpec }, { raw: true });

In [6]:
// Intent diversity: unique intents and their similarity
function cosine(a: number[], b: number[]): number {
  let dot = 0, na = 0, nb = 0;
  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i];
    na += a[i] * a[i];
    nb += b[i] * b[i];
  }
  return dot / (Math.sqrt(na) * Math.sqrt(nb) + 1e-9);
}

const intents = withIntentEmb.map(t => ({
  intent: t.intent ?? '(none)',
  target: t.targetNote,
  embedding: t.intentEmbedding!,
}));

if (intents.length >= 2) {
  console.log('\n=== Intent Embeddings ===');
  console.log(`${intents.length} intents with embeddings\n`);
  
  // Pairwise similarity between intents
  const intentSimData: Array<{intentA: string, intentB: string, similarity: number}> = [];
  for (let i = 0; i < intents.length; i++) {
    for (let j = 0; j < intents.length; j++) {
      intentSimData.push({
        intentA: intents[i].intent.slice(0, 30),
        intentB: intents[j].intent.slice(0, 30),
        similarity: cosine(intents[i].embedding, intents[j].embedding),
      });
    }
  }
  
  const intentSpec = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "title": "Intent Similarity Matrix",
    "width": 400,
    "height": 400,
    "data": { "values": intentSimData },
    "mark": "rect",
    "encoding": {
      "x": { "field": "intentA", "type": "nominal", "title": null },
      "y": { "field": "intentB", "type": "nominal", "title": null },
      "color": { "field": "similarity", "type": "quantitative", "scale": { "scheme": "viridis" } },
      "tooltip": [{ "field": "intentA" }, { "field": "intentB" }, { "field": "similarity", "format": ".3f" }]
    }
  };
  await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": intentSpec }, { raw: true });
} else {
  console.log('\nNot enough intents with embeddings for similarity analysis.');
  console.log('Run more --intent queries to accumulate data.');
}


=== Intent Embeddings ===


19 intents with embeddings



In [7]:
// Trace timeline (if executedAt available)
const withTime = traces.filter(t => t.executedAt).map(t => ({
  time: t.executedAt!,
  target: t.targetNote,
  synthetic: t.synthetic,
  pathLen: t.path.length,
}));

if (withTime.length > 0) {
  console.log('\n=== Trace Timeline ===');
  console.table(withTime.slice(-20)); // last 20
} else {
  console.log('No timestamped traces available.');
}


=== Trace Timeline ===


┌───────┬────────────────────────────┬─────────────────────┬───────────┬─────────┐
│ (idx) │ time                       │ target              │ synthetic │ pathLen │
├───────┼────────────────────────────┼─────────────────────┼───────────┼─────────┤
│     0 │ "2026-03-04T12:33:31.729Z" │ "Hiring Need"       │ false     │       5 │
│     1 │ "2026-03-04T13:41:00.720Z" │ "Over Budget"       │ false     │       5 │
│     2 │ "2026-03-04T13:41:13.470Z" │ "Hiring Need"       │ false     │       5 │
│     3 │ "2026-03-04T13:41:13.474Z" │ "Over Budget"       │ false     │       5 │
│     4 │ "2026-03-04T13:41:13.475Z" │ "Project Costs"     │ false     │       2 │
│     5 │ "2026-03-04T13:43:45.872Z" │ "Over Budget"       │ false     │       5 │
│     6 │ "2026-03-04T13:49:37.525Z" │ "Department Budget" │ false     │       2 │
│     7 │ "2026-03-04T13:49:37.607Z" │ "Department Budget" │ false     │       2 │
│     8 │ "2026-03-04T13:49:37.610Z" │ "Over Budget"       │ false     │       4 │
│   

In [8]:
// Training data quality assessment
console.log('\n=== Training Data Quality ===');

// Duplicate synthetic traces (init re-run problem)
const synPaths = synthetic.map(t => t.path.join(','));
const uniqueSynPaths = new Set(synPaths);
const dupRatio = 1 - uniqueSynPaths.size / Math.max(synPaths.length, 1);
console.log(`Synthetic duplicates: ${synPaths.length - uniqueSynPaths.size}/${synPaths.length} (${(dupRatio * 100).toFixed(0)}%)`);
if (dupRatio > 0.3) {
  console.log('⚠ High duplicate rate — init was likely run multiple times without clearing traces');
}

// Intent embedding coverage
const intentCoverage = withIntentEmb.length / Math.max(real.length, 1);
console.log(`\nReal traces with intent embeddings: ${withIntentEmb.length}/${real.length} (${(intentCoverage * 100).toFixed(0)}%)`);

// Notes never targeted
console.log(`\nNotes never targeted: ${uncovered.length}/${noteNames.size}`);
if (uncovered.length > 0) {
  console.log('  These notes have no training signal — GRU cannot learn to route to them.');
}


=== Training Data Quality ===


Synthetic duplicates: 0/8 (0%)



Real traces with intent embeddings: 19/20 (95%)



Notes never targeted: 6/16


  These notes have no training signal — GRU cannot learn to route to them.


In [9]:
db.close();
console.log('Done');

Done
